In [7]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

f = pd.read_csv("../data/features.csv", index_col="date", parse_dates=True)
f = f.loc[:"2026-06-30"].dropna(subset=["target_big"])

price = ["EUR_absret_1", "EUR_vol_5", "EUR_vol_20", "EUR_maxabs_5", "EUR_vol_ratio",
         "USD_vol_5", "USD_vol_20", "USD_maxabs_5"]
news = [c for c in f.columns
        if c.endswith(("_ratio", "_ratio_3d", "_share")) and not c.startswith(("EUR", "USD"))]
print("Features curs:", len(price), "| Features stiri:", len(news))

Features curs: 8 | Features stiri: 17


In [8]:
def walk_forward(cols, C=1.0):
    preds = []
    for year in range(2019, 2027):
        train = f[f.index.year < year]
        test = f[f.index.year == year]
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
        model.fit(train[cols], train["target_big"])
        preds.append(pd.Series(model.predict_proba(test[cols])[:, 1], index=test.index))
    return pd.concat(preds)

y = f.loc["2019":, "target_big"]
results = {}
for name, cols in {"doar curs": price, "curs + stiri": price + news}.items():
    p = walk_forward(cols)
    results[name] = p
    print(f"{name:14} AUC = {roc_auc_score(y, p):.3f}   AP = {average_precision_score(y, p):.3f}")
print(f"{'aleator':14} AUC = 0.500   AP = {y.mean():.3f}")

doar curs      AUC = 0.866   AP = 0.245
curs + stiri   AUC = 0.836   AP = 0.229
aleator        AUC = 0.500   AP = 0.031


In [9]:
p = results["curs + stiri"]
top = p.sort_values(ascending=False).head(15)
pd.DataFrame({"risc": top.round(3),
              "miscare_mare": y[top.index].astype(int),
              "next_ret_%": f.loc[top.index, "next_ret"].round(2)})

,risc,miscare_mare,next_ret_%
date,,,
2025-05-06,0.992,1,1.22
2025-05-19,0.990,1,1.06
2025-05-20,0.979,0,-0.23
2026-05-04,0.963,1,0.35
2025-05-07,0.923,1,0.45
2025-05-22,0.890,0,-0.03
2025-05-26,0.889,0,0.03
2026-05-06,0.843,0,-0.05
2025-05-29,0.823,0,-0.06


In [10]:
variants = {
    "doar curs": price,
    "curs + 2 stiri": price + ["total_ratio_3d", "politic_ratio_3d"],
    "curs + toate stirile": price + news,
}

for C in [1.0, 0.1]:
    print(f"\nRegularizare C = {C}")
    for name, cols in variants.items():
        p = walk_forward(cols, C=C)
        results[f"{name} C={C}"] = p
        print(f"  {name:22} AUC = {roc_auc_score(y, p):.3f}   AP = {average_precision_score(y, p):.3f}")


Regularizare C = 1.0
  doar curs              AUC = 0.866   AP = 0.245
  curs + 2 stiri         AUC = 0.859   AP = 0.244
  curs + toate stirile   AUC = 0.836   AP = 0.229

Regularizare C = 0.1
  doar curs              AUC = 0.868   AP = 0.257
  curs + 2 stiri         AUC = 0.859   AP = 0.249
  curs + toate stirile   AUC = 0.848   AP = 0.250


In [11]:
both = pd.DataFrame(results)
rank = both.rank(pct=True) * 100
events = rank[y == 1].round(0)
events["next_ret_%"] = f.loc[events.index, "next_ret"].round(2)
events

,doar curs,curs + stiri,doar curs C=1.0,curs + 2 stiri C=1.0,curs + toate stirile C=1.0,doar curs C=0.1,curs + 2 stiri C=0.1,curs + toate stirile C=0.1,next_ret_%
date,,,,,,,,,
2019-01-22,83.0,50.0,83.0,82.0,50.0,85.0,84.0,73.0,0.91
2019-01-30,95.0,90.0,95.0,94.0,90.0,95.0,94.0,95.0,-0.44
2019-01-31,97.0,96.0,97.0,97.0,96.0,97.0,97.0,96.0,0.36
2019-02-26,80.0,43.0,80.0,72.0,43.0,81.0,72.0,44.0,-0.44
2019-06-27,79.0,71.0,79.0,83.0,71.0,83.0,86.0,76.0,0.27
2020-02-20,53.0,50.0,53.0,60.0,50.0,56.0,62.0,55.0,0.40
2020-03-16,80.0,80.0,80.0,78.0,80.0,84.0,82.0,82.0,0.43
2021-03-29,66.0,31.0,66.0,69.0,31.0,56.0,59.0,33.0,0.38
2021-03-31,82.0,79.0,82.0,83.0,79.0,79.0,81.0,78.0,-0.27


In [12]:
cols = price + ["total_ratio_3d", "politic_ratio_3d"]
train = f[f.index.year < 2025]
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000))
model.fit(train[cols], train["target_big"])

coefs = pd.Series(model[-1].coef_[0], index=cols).sort_values()
coefs.round(3)

EUR_vol_5          -0.112
politic_ratio_3d   -0.071
EUR_vol_ratio      -0.041
USD_maxabs_5        0.007
total_ratio_3d      0.027
USD_vol_5           0.126
USD_vol_20          0.172
EUR_absret_1        0.250
EUR_maxabs_5        0.298
EUR_vol_20          0.413
dtype: float64